# Rizzler V24 — Real Gemini / Multi-Agent Qualification (Colab and Kaggle)
Run in **Google Colab or Kaggle** with an authorized **GEMINI_API_KEY**. It checks the pinned source and runs offline tests, real model search, artifact generation, SQL/Excel and final verification. Reports are exported even on failed jobs. **Passing a notebook does not certify production-grade content or infrastructure.**

In [ ]:
import os, sys, json, subprocess, pathlib, hashlib, time, secrets, shutil
ROOT = pathlib.Path("/kaggle/working" if pathlib.Path("/kaggle/working").is_dir() else "/content")
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / "Rizzler-V24"
BRANCH = "feature/v24-context-convergence-and-safe-repair"
URL = "https://github.com/SAUNAK359/Voice-AI.git"
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--force", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, URL, str(REPO)], check=True)
SHA = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Qualified source revision:", SHA)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                str(REPO) + "[work,test]", "pymupdf>=1.25,<2"], check=True)
sys.path.insert(0, str(REPO / "src"))
OUT = ROOT / "rizzler_v24_evidence"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "source_sha.txt").write_text(SHA + "\n", encoding="utf-8")
print("Notebook environment:", ROOT)

## Offline regression isolation
Run authorization, context recovery and existing specialist contract tests without live credentials. These are source-code tests, not an authenticated Gemini qualification.

In [ ]:
import copy
TEST_ENV={k:v for k,v in os.environ.items() if not k.startswith("RIZZLER_") and k not in ("GEMINI_API_KEY","GOOGLE_API_KEY")}
TEST_ENV.update(PYTHONPATH=str(REPO/"src"))
tests=["test_v24_context_convergence.py","test_v24_intent_and_recovery.py","test_v23_work_studio.py","test_v23_outcome_contract.py","test_v23_sql_exploration.py"]
runs={}
for filename in tests:
    p=subprocess.run([sys.executable,"-m","unittest","discover","-s",str(REPO/"tests"),"-p",filename,"-v"],cwd=REPO,env=TEST_ENV,text=True,capture_output=True)
    (OUT/(filename+".log")).write_text(p.stdout+"\n"+p.stderr)
    runs[filename]={"exit_code":p.returncode,"passed":p.returncode==0}
print(json.dumps(runs,indent=2))
(OUT/"offline_checks.json").write_text(json.dumps(runs,indent=2))

## Authenticated Gemini preflight
Set a **GEMINI_API_KEY** secret in Colab (key icon) or Kaggle (Add-ons → Secrets). Grant the notebook access; do not paste the key into a cell. `RIZZLER_GEMINI_MODEL` may be configured explicitly. Any provider failure is reported and exported, never silently disabled.

In [ ]:
import json, os, secrets
from google import genai
from rizzler.config import Settings
key = os.environ.get("GEMINI_API_KEY", "").strip()
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("GEMINI_API_KEY") or ""
    except (ImportError, RuntimeError, OSError, KeyError):
        pass
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY") or ""
    except (ImportError, RuntimeError, OSError, KeyError):
        pass
if not key:
    raise RuntimeError("Add Gemini API key as a Colab or Kaggle secret named GEMINI_API_KEY")
os.environ["GEMINI_API_KEY"] = key
os.environ["RIZZLER_WORK_ROOT"] = str(OUT / "work")
os.environ["RIZZLER_ACCESS_KEY"] = secrets.token_urlsafe(32)
os.environ["RIZZLER_WORK_SIGNING_KEY"] = secrets.token_urlsafe(48)
os.environ["RIZZLER_V23_MODE"] = "full"
os.environ["RIZZLER_CONTEXT_CONVERGENCE"] = "1"
os.environ["RIZZLER_GEMINI_NATIVE_TOOLS"] = "1"
os.environ["RIZZLER_GEMINI_ENABLE_THINKING"] = "1"
os.environ["RIZZLER_ARTIFACT_QA"] = "strict"
cfg = Settings.from_env()
preflight = {"model": cfg.gemini_model, "passed": False, "source_sha": SHA}
try:
    with genai.Client(api_key=key) as client:
        rsp = client.models.generate_content(
            model=cfg.gemini_model,
            contents="Return the exact token: RIZZLER_V24_PREFLIGHT_OK")
        preflight["passed"] = "RIZZLER_V24_PREFLIGHT_OK" in (rsp.text or "")
        if not preflight["passed"]:
            preflight["failure"] = "unexpected_model_response"
except Exception as exc:
    preflight["failure"] = type(exc).__name__
(OUT / "gemini_preflight.json").write_text(json.dumps(preflight, indent=2), encoding="utf-8")
print("Gemini preflight:", preflight)
if not preflight["passed"]:
    raise RuntimeError("Live Gemini model preflight failed; see gemini_preflight.json")

## Strict render dependencies
PDF, PowerPoint and Word strict visual QA require LibreOffice and PyMuPDF. Do not silently turn off strict QA if these are unavailable.

In [ ]:
if shutil.which("libreoffice") is None:
    subprocess.run(["apt-get","update","-qq"],check=True)
    subprocess.run(["apt-get","install","-y","-qq","libreoffice-writer","libreoffice-impress"],check=True)
import fitz
print("Strict visual QA dependencies available:",shutil.which("libreoffice"),"PyMuPDF",fitz.__doc__.splitlines()[0])

## Live research → PowerPoint, Word, PDF and ZIP
Run the real worker with top-level `await`. A worker returning `True` means **it processed a job**, not that the job completed. Records are persisted even on failure, including owner-scoped diagnostics and research gaps.

In [ ]:
import json, traceback
from rizzler.work_store import WorkStore
from rizzler.work_worker import WorkWorker

store = WorkStore(OUT / "work")
owner = "kaggle_qualification_owner"
objective = (
    "Investigate the officially documented capabilities and limitations of "
    "Google Gemini API grounding, URL Context and Code Execution. Use "
    "verifiable first-party documentation, identify which combinations "
    "require qualification, and produce an evidence-linked stakeholder PPT, "
    "Word report, PDF report and source package. Do not invent facts, limits "
    "or citations."
)
job = store.submit(owner, objective, ["research","pptx","docx","pdf","zip"],
                   "v24-kaggle-real-gemini-001", [],
                   requested_formats=["pptx","docx","pdf","zip"])
job_id = job["id"]
worker_error = None
try:
    for _ in range(4):
        if store.get(owner, job_id)["status"] != "queued":
            break
        handled = await WorkWorker(store).once()
        if not handled:
            break
except Exception as exc:
    worker_error = {"type": type(exc).__name__}
    traceback.print_exc()

result = store.get(owner, job_id)
steps = store.agent_steps(owner, job_id)
artifacts = store.artifacts(owner, job_id)
diagnostics = store.diagnostics(owner, job_id)
events = store.events(owner, job_id)
cognitive_events = store.cognitive_events(owner, job_id, limit=200)
record = {"job": result, "steps": steps, "artifacts": artifacts,
          "diagnostics": diagnostics, "events": events,
          "cognitive_events": cognitive_events, "worker_error": worker_error}
(OUT / "live_job.json").write_text(json.dumps(record, indent=2, default=str),
                                    encoding="utf-8")
print("Live job:", result["status"], "| artifacts:", [x["kind"] for x in artifacts])
for step in steps:
    print(step["agent"], step["status"], step.get("error_code",""))
if diagnostics:
    print("SAFE diagnostics:", json.dumps(diagnostics[-5:], indent=2, default=str)[:6000])

## Visual, numerical and source QA
This independent pass opens each generated format and checks that every requested artifact exists, is structurally valid and has a digest. It does not substitute for a human factual/visual review.

In [ ]:
import json
from rizzler.artifacts import verify_artifact
required = {"pptx", "docx", "pdf", "zip"}
state = store.get(owner, job_id)
records = store.artifacts(owner, job_id)
observed = {record["kind"] for record in records}
verified, problems = [], []
if state["status"] == "completed":
    for record in records:
        try:
            path, metadata = store.artifact(owner, record["id"])
            receipt = verify_artifact(path, record["kind"])
            assert receipt.verified and receipt.sha256 == record["sha256"]
            verified.append({"kind": record["kind"], "sha256": receipt.sha256})
        except Exception as exc:
            problems.append({"kind": record["kind"], "error_type": type(exc).__name__})
else:
    problems.append({"category": "parent_job_not_completed",
                     "status": state["status"]})
quality = store.quality_report(owner, job_id)
gate = {"required": sorted(required), "observed": sorted(observed),
        "verified": verified, "missing": sorted(required-observed),
        "problems": problems, "job_status": state["status"], "quality": quality}
gate["passed"] = (state["status"] == "completed" and
                  not problems and not gate["missing"] and
                  quality.get("quality") is not None)
(OUT / "artifact_gate.json").write_text(json.dumps(gate, indent=2, default=str),
                                         encoding="utf-8")
print("Artifact QA:", "PASS" if gate["passed"] else "NOT QUALIFIED")
print(json.dumps({k:gate[k] for k in ("job_status","missing","problems")}, indent=2))

## Live SQL → Excel → PowerPoint coordination
A separate job uses a clearly labeled synthetic dataset, so the expected calculations can be checked without inventing market facts. This exercises SQL analysis and professional spreadsheet generation together.

In [ ]:
import csv, io
dataset = ("quarter,imports,exports,region\n"
           "Q1,125,95,North\nQ2,140,105,North\n"
           "Q3,160,117,South\nQ4,175,133,South\n").encode("utf-8")
rows = list(csv.DictReader(io.StringIO(dataset.decode("utf-8"))))
expected = {
    "total_imports": sum(int(row["imports"]) for row in rows),
    "total_exports": sum(int(row["exports"]) for row in rows),
}
expected["net_balance"] = expected["total_exports"]-expected["total_imports"]

# Re-running this cell keeps the original queued job and avoids duplicate uploads.
if "job2" not in globals():
    upload = store.register_upload(owner, "synthetic_trade_qa.csv", dataset)
    objective2 = (
        "Analyze this attached SYNTHETIC quarterly dataset. Calculate "
        "total imports, total exports, per-quarter balances and net balance, "
        "with source-aware methodology, limitations, editable workbook "
        "formulas and a professional PowerPoint. The dataset is fictional."
    )
    job2 = store.submit(owner, objective2, ["xlsx","pptx","zip"],
                        "v24-kaggle-sql-excel-001", [upload["id"]],
                        requested_formats=["xlsx","pptx","zip"])

worker2_error = None
try:
    for _ in range(4):
        if store.get(owner, job2["id"])["status"] != "queued":
            break
        if not await WorkWorker(store).once():
            break
except Exception as exc:
    worker2_error = {"type": type(exc).__name__}

result2 = store.get(owner, job2["id"])
steps2 = store.agent_steps(owner, job2["id"])
files2 = store.artifacts(owner, job2["id"])
diagnostics2 = store.diagnostics(owner, job2["id"])
events2 = store.events(owner, job2["id"])
needed = {"xlsx","pptx","zip"}
sql_excel = {
    "job": result2, "expected_from_synthetic_dataset": expected,
    "steps": steps2, "artifacts": files2, "diagnostics": diagnostics2,
    "events": events2, "worker_error": worker2_error,
    "missing": sorted(needed-{f["kind"] for f in files2}),
}
sql_excel["passed"] = (result2["status"] == "completed" and not sql_excel["missing"])
(OUT/"sql_excel_job.json").write_text(json.dumps(sql_excel,indent=2,default=str),
                                     encoding="utf-8")
print("Analytical job:", result2["status"], "| qualified:", sql_excel["passed"])
for step in steps2:
    print(step["agent"], step["status"], step.get("error_code",""))
if diagnostics2:
    print("SAFE diagnostics:",json.dumps(diagnostics2[-5:],indent=2,default=str)[:6000])

## Export full test evidence
The ZIP contains logs, job receipts, qualification decisions and SHA. It does **not** include your Gemini API key.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
target = ROOT / ("Rizzler_V24_Qualification_" + SHA[:12] + ".zip")
qualification = {
    "source_sha": SHA, "gemini_preflight": preflight["passed"],
    "offline_passed": all(item["passed"] for item in runs.values()),
    "research_artifacts_passed": gate["passed"],
    "sql_excel_passed": sql_excel["passed"],
}
qualification["passed"] = all(qualification[k] for k in
                             ("gemini_preflight","offline_passed",
                              "research_artifacts_passed","sql_excel_passed"))
(OUT / "qualification.json").write_text(json.dumps(qualification, indent=2),
                                         encoding="utf-8")
with ZipFile(target, "w", ZIP_DEFLATED) as archive:
    for item in OUT.rglob("*"):
        if item.is_file() and "uploads" not in item.parts and item.suffix not in (".db",".sqlite3"):
            archive.write(item, item.relative_to(OUT))
print("Qualification:", "PASS" if qualification["passed"] else "NOT QUALIFIED")
print("Evidence bundle:", target)
print("Source revision:", SHA)
print("Upload the ZIP to the conversation to investigate any failed stage.")